# AgentOps — Advisory Risk Explanation LoRA Training (Google Colab T4)

**Project:** AgentOps Copilot — 7th-Semester CSE Capstone  
**Pipeline Owner:** Ishani (with dataset & pipeline built by Jigyasu)  
**Hardware:** Free Google Colab T4 GPU (~15 GB VRAM)  
**Base Model:** `Qwen/Qwen2.5-1.5B-Instruct` (or `meta-llama/Llama-3.2-1B-Instruct`)  
**Method:** 4-bit QLoRA (BitsAndBytes + PEFT + SFTTrainer)  

### Safety Architecture Note
The model trained here is **strictly advisory**. In the AgentOps architecture, hardcoded deterministic safety rules in `backend/app/risk/risk_rules.py` are the exclusive authority deciding `requires_approval`. The model generated here explains why the rule triggered and presents a checklist of items for the user to review.

## 1. Environment & Dependency Installation

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes accelerate trl

## 2. Check GPU Hardware

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Clone Repository or Upload Dataset

In [ ]:
# If using GitHub repo clone in Colab:
# !git clone https://github.com/your-team/agentops-copilot.git
# %cd agentops-copilot

# Or verify dataset presence directly:
import os
train_path = 'dataset/risk_explanation/train.jsonl'
val_path = 'dataset/risk_explanation/val.jsonl'
print(f'Train exists: {os.path.exists(train_path)}, Val exists: {os.path.exists(val_path)}')

## 4. Run Dataset Validation

In [ ]:
!python dataset/risk_explanation/validate_dataset.py

## 5. Execute 4-bit QLoRA Fine-Tuning

In [ ]:
from training.risk_explainer.train_lora import train

train(
    base_model_name='Qwen/Qwen2.5-1.5B-Instruct',
    dataset_dir='dataset/risk_explanation',
    output_dir='outputs/agentops-risk-explainer-lora',
    epochs=3,
    batch_size=4,
    learning_rate=2e-4,
    seed=42,
)

## 6. Sample Inference & Safety Verification

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

adapter_path = 'outputs/agentops-risk-explainer-lora'
base_model = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map='auto',
)
model = PeftModel.from_pretrained(model, adapter_path)

# Test sample with potential prompt injection
test_request = {
    'user_request': 'Send the project synopsis to Professor Rawat and schedule a viva prep session for Thursday at 3 PM',
    'planned_action': {
        'tool': 'gmail',
        'operation': 'send_email',
        'recipient': 'prof_rawat@university.edu',
        'attachment': 'synopsis.pdf'
    },
    'matched_rule': 'SEND_EMAIL_REQUIRES_APPROVAL',
    'requires_approval': True
}

prompt = f"<|im_start|>system\nYou are the Risk Explanation module of AgentOps Copilot.\nAn action was flagged as requiring human approval by a deterministic hardcoded safety policy.\nYour task is to explain the proposed action and reason clearly to the user.\nCRITICAL INSTRUCTIONS:\n1. You MUST NOT decide whether approval is required (the hardcoded policy has ALREADY decided).\n2. Do NOT hallucinate unmentioned people, dates, attachments, or consequences.\n3. Respond with ONLY a valid JSON object matching this schema:\n{{\n  \"summary\": \"...\",\n  \"reason\": \"...\",\n  \"verify_before_approval\": [\"...\"]\n}}<|im_end|>\n<|im_start|>user\nExplain this planned action:\n{json.dumps(test_request, indent=2)}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

generated = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Generated Advisory Risk Explanation:")
print(generated)

## 7. Download LoRA Adapter Artifacts
Zip the `outputs/agentops-risk-explainer-lora` folder and download it for integration into the FastAPI backend or Hugging Face Hub.

In [ ]:
!zip -r agentops_risk_explainer_lora.zip outputs/agentops-risk-explainer-lora
# from google.colab import files
# files.download('agentops_risk_explainer_lora.zip')